# S-conjugacy — spheres on the faces of an S-conjugate quad mesh

Same pipeline idea as `Lconjugacy` (Laguerre_project), with spheres in place of
oriented planes. The logic lives in `moebius/sconjugacy.py`; this notebook sets
the parameters, runs each stage and shows the result in the kayviz viewer.

| Stage | What it does |
|---|---|
| 0 | load a B-spline from `data/bsplines/`, choose the (u, v) range and sampling |
| 0b | optional: deform the B-spline so that H ≠ 0 (r = 1/H needs it) |
| 1 | radius function: r = λ r_H with r_H = 1/H (mean-curvature sphere), λ ∈ [0, 2], fitted by a spline r(u, v) |
| 2 | principal frame at every sample, a first direction t1 and its S-conjugate t2 |
| 3 | frame field → `quadRemesher` → quad mesh Q, then glide + fairness → Q' |
| 4 | Q' projected onto the B-spline; one sphere per face from the barycentre's (u, v) — the sphere mesh has the dual combinatorics |
| 5 | post-optimisation: E_sphere, E_unit, E_reg, E_fair, E_prox_f, E_prox_C (no support structure) |
| 6 | save OBJ files and the state to `notebooks/out/sconjugacy/<surface>/<experiment>/` |

**S-conjugacy.** With t1 = a1 e1 + a2 e2 and t2 = b1 e1 + b2 e2 in the principal frame
and r the sphere radius at the point,

$$(1 - r\kappa_1)\, a_1 b_1 + (1 - r\kappa_2)\, a_2 b_2 = 0 .$$

Given t1 at angle θ to e1, t2 = J M a with M = diag(1 − rκ1, 1 − rκ2) and J the rotation by 90°.
Two special cases are useful when choosing θ and λ:

- **λ = 0** (r = 0): M = I, so t2 ⟂ t1 — the orthogonal net.
- **λ = 1** (r = 1/H): M is a multiple of diag(1, −1), so the partner of θ is **90° − θ**.
  θ = 45° is then self-conjugate (t2 = t1) everywhere, and the frame field is degenerate.

In [1]:
import sys, os
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

CHAKANA = (ROOT / "Chakana_Geo").resolve()
for sub in ["hanan/src", "kayviz/src"]:
    p = str(CHAKANA / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

%load_ext autoreload
%autoreload 2

import numpy as np
import kayviz as kv
import ipywidgets as widgets
from IPython.display import display

import moebius.sconjugacy as sc
from moebius.pipeline import (
    check_remesher, remesh_quality, fair_remeshed_surface,
    setup_bspline_optimizer, run_bspline_optimizer,
)
from moebius.utils.bsplines import sample_bspline_surface, bspline_curvatures
from moebius.utils.spheres import implicit_sphere_from_center_radius, center_radius_from_implicit

kv.init()
kv.clear()

[kayviz] Viewer running at http://127.0.0.1:8000


In [13]:
# ── viewer helpers ────────────────────────────────────────────────────────────
def show_surface(st, name="surface"):
    V, F = sample_bspline_surface(st.bspline, st.u_pts, st.v_pts)
    K, H, _ = bspline_curvatures(st.bspline, st.u_pts, st.v_pts)
    kv.register_surface_mesh(name, V, F, color=(0.7, 0.7, 0.7), show_edges=True)
    kv.add_scalar_quantity(name, "H", H.ravel(), defined_on="vertices")
    kv.add_scalar_quantity(name, "K", K.ravel(), defined_on="vertices")
    return V, F

def mean_edge(st):
    V, F = sample_bspline_surface(st.bspline, st.u_pts, st.v_pts)
    return np.mean(np.linalg.norm(V[F[:, 1]] - V[F[:, 0]], axis=1))

def show_cross(st, scale=None):
    P = st.bspline(st.u_pts, st.v_pts).reshape(-1, 3)
    s = 1.2 * mean_edge(st) if scale is None else scale
    kv.register_curve_network("t1", *sc.cross_segments(P, st.t1, s), radius=0.0008, color=(0.9, 0.2, 0.1))
    kv.register_curve_network("t2 (S-conjugate)", *sc.cross_segments(P, st.t2, s), radius=0.0008, color=(0.1, 0.3, 0.9))

def show_sphere_mesh(st, V=None, A=None, B=None, C=None, panels=True, name="spheres"):
    # Sphere centres on Q's faces, joined with the dual combinatorics, and the panels
    if A is None:
        A, B, C = implicit_sphere_from_center_radius(st.c_face, st.r_face)
    V = st.V_remesh if V is None else V
    c, r, _ = center_radius_from_implicit(A, np.asarray(B).reshape(-1, 3), C)
    kv.register_surface_mesh(f"{name}: centres (dual mesh)", c, st.sphere_dual_faces,
                             color=(0.6, 0.02, 0.8), show_edges=True, opacity=0.6)
    if panels:
        from moebius.glyphs import panels_from_spheres     # needs the 'triangle' package
        pV, pF = panels_from_spheres(V, st.F_remesh_quads, A, np.asarray(B).reshape(-1, 3), C,
                                     target_triangles=120)
        kv.register_surface_mesh(f"{name}: panels", pV, pF, color=(0.85, 0.75, 0.4))

---
## 0 — Load the B-spline, choose the (u, v) range

Any file of `data/bsplines/` (without `.json`). The sliders resample live; the
surface is coloured by H and K. Re-run stage 1 after changing the range.

In [3]:
# ── parameters (edit these) ───────────────────────────────────────────────────
SURFACE_NAME    = "Tunel"
EXPERIMENT_NAME = "experiment0"

U_SAMPLE   = 30
V_SAMPLE   = 30
INTERVAL_U = [0.1, 0.9]
INTERVAL_V = [0.1, 0.9]

FLIP_NORMALS = False   # reverse u (flips the normals, so the sign of H and r)
SWAP_UV      = False   # exchange u and v (also flips the normals)
# ─────────────────────────────────────────────────────────────────────────────

st = sc.load_sconj_surface(SURFACE_NAME, EXPERIMENT_NAME,
                           u_sample_num=U_SAMPLE, v_sample_num=V_SAMPLE,
                           interval_u=INTERVAL_U, interval_v=INTERVAL_V)
if FLIP_NORMALS:
    sc.flip_normals(st)
if SWAP_UV:
    sc.swap_uv(st)

def _update_surface(u_rng, v_rng, u_n, v_n):
    sc.sample_selection(st, interval_u=list(u_rng), interval_v=list(v_rng),
                        u_sample_num=u_n, v_sample_num=v_n)
    kv.clear()
    show_surface(st)
    _, H, _ = bspline_curvatures(st.bspline, st.u_pts, st.v_pts)
    info.value = (f"{st.name}: {u_n}x{v_n} samples   H in [{H.min():.3f}, {H.max():.3f}]"
                  + ("   WARNING: H changes sign" if (H > 0).any() and (H < 0).any() else ""))

_sl = dict(continuous_update=False, style={"description_width": "80px"}, layout=widgets.Layout(width="420px"))
u_w  = widgets.FloatRangeSlider(value=st.interval_u, min=0, max=1, step=0.01, description="u range", **_sl)
v_w  = widgets.FloatRangeSlider(value=st.interval_v, min=0, max=1, step=0.01, description="v range", **_sl)
nu_w = widgets.IntSlider(value=st.u_sample_num, min=5, max=100, description="u samples", **_sl)
nv_w = widgets.IntSlider(value=st.v_sample_num, min=5, max=100, description="v samples", **_sl)
info = widgets.Label()
out = widgets.interactive_output(_update_surface, {"u_rng": u_w, "v_rng": v_w, "u_n": nu_w, "v_n": nv_w})
display(widgets.VBox([widgets.HBox([u_w, nu_w]), widgets.HBox([v_w, nv_w]), info]), out)
kv.show()

Output()

[kayviz] Reusing the open viewer tab — http://127.0.0.1:8000/?app=script


---
## 0b — Optional: B-spline optimisation (H ≠ 0)

r = λ/H blows up where H = 0. The surfaces kept in `data/bsplines/` satisfy H ≠ 0;
for another surface, or a range where H changes sign, this deforms the control
points (the Moebius stage 1b). Off by default.

In [ ]:
RUN_BSPLINE_OPT = False
if RUN_BSPLINE_OPT:
    setup_bspline_optimizer(st, H_thresh=0.05, H_max=None, w_H=1.0, w_step=0.1)
    run_bspline_optimizer(st, max_iter=30)
    kv.clear()
    show_surface(st)
    kv.show()

---
## 1 — Radius function r = λ r_H

r_H = 1/H is the radius of the mean-curvature sphere (centre on the normal at
distance r). The slider scales it, r = λ r_H with λ ∈ [0, 2]; each change refits
the spline r(u, v) (cubic, interpolating the samples) used by every later
stage. The viewer shows the centre surface S + r n coloured by r.

In [17]:
LAMBDA = 0.8

def _update_radius(lam):
    sc.compute_radius(st, lam=lam)
    kv.clear()
    show_surface(st)
    Vc, F = sc.sphere_center_surface(st)
    kv.register_surface_mesh("sphere centres S + r n", Vc, F, color=(0.6, 0.02, 0.8), opacity=0.6)
    kv.add_scalar_quantity("sphere centres S + r n", "r", sc.radius_grid(st), defined_on="vertices")
    r = st.r_H
    rinfo.value = (f"lam = {st.lam:.2f}:  r in [{r.min():.4f}, {r.max():.4f}]   "
                   f"spline fit error {st.r_fit_err:.1e}")

lam_w = widgets.FloatSlider(value=LAMBDA, min=0.0, max=2.0, step=0.05, description="λ", **_sl)
rinfo = widgets.Label()
out = widgets.interactive_output(_update_radius, {"lam": lam_w})
display(widgets.VBox([lam_w, rinfo]), out)
kv.show()

Output()

[kayviz] Viewer already open — http://127.0.0.1:8000/?app=script


---
## 2 — Principal frame and S-conjugate directions

At every sample the principal frame (e1, e2, κ1 ≤ κ2) is computed, the first
direction is t1 = cos θ e1 + sin θ e2, and t2 is its S-conjugate for the fitted
r. `method`:

- `constant` — θ everywhere;
- `linearU` / `linearV` — θ goes linearly from `θ start` to `θ end` across the u / v range.

The printout gives the S-conjugacy residual (should be ~1e-16) and the angle
between t1 and t2. Near 0° the frame is degenerate (t1 almost self-conjugate),
and the remesher cannot follow it — change θ or λ. At λ = 1 avoid θ = 45°.

In [18]:
def _update_dirs(method, theta, theta_rng):
    sc.compute_directions(st, method=method, theta_deg=theta, theta_range=theta_rng)
    kv.clear()
    show_surface(st)
    ang = sc._frame_angle_deg(st.t1, st.t2)
    kv.add_scalar_quantity("surface", "angle(t1,t2)", ang, defined_on="vertices")
    show_cross(st)

method_w = widgets.Dropdown(options=["constant", "linearU", "linearV"], value=st.theta_method,
                            description="method", style={"description_width": "80px"})
theta_w  = widgets.FloatSlider(value=st.theta_deg, min=0, max=90, step=1, description="θ (deg)", **_sl)
trange_w = widgets.FloatRangeSlider(value=st.theta_range, min=0, max=90, step=1,
                                    description="θ start/end", **_sl)
out = widgets.interactive_output(_update_dirs, {"method": method_w, "theta": theta_w, "theta_rng": trange_w})
display(widgets.VBox([method_w, theta_w, trange_w]), out)
kv.show()

Output()

[kayviz] Viewer already open — http://127.0.0.1:8000/?app=script


---
## 3 — Remeshing

`export_sconjugate_field` writes into `st.save_path` the triangulated sampling grid
and one (t1, t2) frame per quad, evaluated at the quad's parameter midpoint
(`remesh_input.obj`, `D1.dat`, `D2.dat`, `indices.dat`, as in the Moebius pipeline).
Degenerate frames (angle < 5°) and frames far from their neighbours are left
out; the remesher interpolates them.

**The binary.** `bin/quadRemesher` is a Linux x86-64 build. On macOS point
`st.remesher_bin` to a build with the flag CLI (`--field`, `--gradient`, `--batch`),
or `export REMESHER_BIN=...` before starting Jupyter.

`GRADIENT`: higher = denser quads. `CONSTRAINTS` ~0.1 lets the solver smooth the
field (fewer singularities). The post-optimisation needs an all-quad mesh.

In [19]:
# st.remesher_bin = "/path/to/quadRemesher"
field = sc.export_sconjugate_field(st, one_frame_per_quad=True, filter_outliers=True,
                                   max_neighbour_deg=35.0, min_frame_angle_deg=5.0)
check_remesher(st.remesher_bin or None)

Exported 900 vertices, 1682 triangles -> /Users/cisneras/KAUST/Moebius_project/notebooks/out/sconjugacy/Tunel/experiment0
  841 frames constrained (841 quads)
  angle(t1,t2)  min 49.6  mean 53.0  max 61.5 deg
  in-plane rotation  mean 0.1  max 0.7 deg


OSError: [Errno 8] Exec format error: '/Users/cisneras/KAUST/Moebius_project/bin/quadRemesher'

In [ ]:
GRADIENT    = 40.0
CONSTRAINTS = 0.1
EXTRACTOR   = "qex"

res = sc.remesh(st, gradient=GRADIENT, constraints=CONSTRAINTS, extractor=EXTRACTOR, batch=True)
# res = sc.remesh(st, gradient=GRADIENT, constraints=CONSTRAINTS, extractor=EXTRACTOR,
#                 batch=False)    # remesher viewer: '+'/'-' gradient, 'R' preview, 'S' save
if st.V_remesh is not None:
    kv.clear()
    show_surface(st)
    kv.register_surface_mesh("Q (remeshed)", st.V_remesh, st.F_remesh, color=(0.9, 0.8, 0.1), show_edges=True)
    kv.show()

In [ ]:
# ── other options ─────────────────────────────────────────────────────────────
# rows = sc.sweep_gradient(st, [20, 40, 80], constraints=CONSTRAINTS, extractor=EXTRACTOR)
# sc.load_remeshed(st)                          # <save_path>/<name>_Remeshed.obj
# sc.load_remeshed(st, "/path/to/other_Remeshed.obj")

---
## 3b — Fair the remeshed mesh (Q → Q')

As in the Moebius pipeline: the mesh glides over the reference surface while
the fairness of the parameter lines removes the zigzag of the remesher output
(`pipeline.fair_remeshed_surface`: proximity 5, fairness 0.01 damped to 0, edge
lengths held, boundary fixed). Updates `st.V_remesh` in place.

In [ ]:
FAIR_MAX_ITER     = 20
W_PROX_FAIR       = 5.0
W_GLIDE_FAIR      = 5.0
W_FAIR_FAIR       = 1e-2
W_EDGE_FAIR       = 1e-1
FIX_BOUNDARY_FAIR = True

V_raw = st.V_remesh.copy()
fair_remeshed_surface(st, max_iter=FAIR_MAX_ITER, w_prox=W_PROX_FAIR, w_glide=W_GLIDE_FAIR,
                      w_fair=W_FAIR_FAIR, w_edge=W_EDGE_FAIR, fix_boundary=FIX_BOUNDARY_FAIR)

kv.clear()
kv.register_surface_mesh("Q raw", V_raw, st.F_remesh, show_edges=True, show=False)
kv.register_surface_mesh("Q' fair", st.V_remesh, st.F_remesh, show_edges=True)
kv.add_scalar_quantity("Q' fair", "moved", np.linalg.norm(st.V_remesh - V_raw, axis=1), defined_on="vertices")
kv.show()

---
## 4 — Sphere mesh initialisation

1. The vertices of Q' are projected onto the B-spline (closest point, Gauss–Newton
   on the parameters).
2. The barycentre of each face f is projected onto the B-spline, giving (u_f, v_f).
3. The sphere s_f has centre B(u_f, v_f) + r(u_f, v_f) n(u_f, v_f) and radius
   r(u_f, v_f), with r the radius function fitted in stage 1.

One sphere per face of Q, so the sphere mesh has the dual combinatorics: its
vertices are the faces of Q, its faces the rings of faces around the inner vertices.

In [ ]:
sc.project_mesh_onto_bspline(st)
sc.init_sphere_mesh(st)

err0 = sc.sphere_fit_error(st)
print(f"initial |dist - r|: median {np.median(err0):.3e}  max {err0.max():.3e}")

kv.clear()
kv.register_surface_mesh("Q'", st.V_remesh, st.F_remesh, color=(0.9, 0.8, 0.1), show_edges=True)
kv.add_scalar_quantity("Q'", "|dist - r| (init)", err0, defined_on="faces")
show_sphere_mesh(st, name="init")
kv.show()

---
## 5 — Post-optimisation

The Moebius post-optimisation without the support structure: the sphere
centres need not form planar rings, so E_supp and E_torsal_plane are dropped,
and with them the variables nd, l, n_l. A few sphere-only warm-up steps run
first, with the vertices held.

E_reg (step dampening) is 0 by default, as in the Moebius notebook: with
`adaptive_mu=False` every step is accepted and its residual is identically zero.

In [ ]:
# ── parameters ────────────────────────────────────────────────────────────────
POSTOPT_MAX_ITER = 30
WARMUP_ITERS     = 10

W_SPHERE   = 1.0     # E_sphere : spheres pass through their face vertices
W_UNIT     = 10.0    # E_unit   : <B,B> - 4AC = 1
W_FAIR     = 1e-1    # E_fair   : quad-mesh fairness
W_PROX     = 1e-4    # E_prox_f : vertices -> reference surface (point)
W_GLIDE    = 1e-4    # E_prox_f : vertices -> reference surface (tangent plane)
W_PROX_C   = 1e-5    # E_prox_C : sphere centres -> centre surface S + r n
W_REG_V    = 0.0     # E_reg    : vertices
W_REG_S    = 0.0     # E_reg    : sphere coefficients A, B, C
FIX_BOUNDARY = True
# ─────────────────────────────────────────────────────────────────────────────

sc.setup_sconj_postopt(st, w_sphere=W_SPHERE, w_unit=W_UNIT, w_fair=W_FAIR,
                       w_prox=W_PROX, w_glide=W_GLIDE, w_prox_c=W_PROX_C,
                       w_reg_v=W_REG_V, w_reg_s=W_REG_S,
                       warmup_iters=WARMUP_ITERS, fix_boundary=FIX_BOUNDARY)
sc.run_sconj_postopt(st, max_iter=POSTOPT_MAX_ITER)

err = sc.sphere_fit_error(st)
print(f"\n|dist - r|: median {np.median(err):.3e}  max {err.max():.3e}")

In [ ]:
# ── visualise the result ──────────────────────────────────────────────────────
V_opt = st.postopt.unpack("v").reshape(-1, 3)
A, B, C = st.postopt.unpack("A", "B", "C")

kv.clear()
kv.register_surface_mesh("optimised mesh", V_opt, st.F_remesh, show_edges=True, show=False)
kv.add_scalar_quantity("optimised mesh", "|dist - r|", err, defined_on="faces")
show_sphere_mesh(st, V=V_opt, A=A, B=B, C=C, name="optimised")
kv.show()

---
## 6 — Save results

Writes `surface.obj`, `sphere_centers.obj` (centre surface S + r n),
`remeshed.obj`, `sphere_mesh_centers.obj` (dual mesh of the sphere centres),
`optimised_mesh.obj`, `sphere_panels.obj` and `state.pickle`.

In [ ]:
sc.save_sconj_results(st)
# st = sc.load_state(...)  ->  from moebius.pipeline import load_state